# 01 — Load Market Sales (Parameterized)
## Goal
- read project config
- define widgets
- load selected sales data
- print useful run context

In [0]:
import yaml
from pyspark.sql import functions as F

In [0]:
config_path = "/Workspace/Repos/adb-tetiana@startsteps.org/beiersdorf-live-demo/config/project_config.yml"

In [0]:
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

In [0]:
dbutils.widgets.text(name="report_start_date", defaultValue="2026-04-25", label="Report Start Date")
dbutils.widgets.text(name="report_end_date", defaultValue="2026-04-30", label="Report End Date")

In [0]:
dbutils.widgets.dropdown(name= "market", defaultValue= "ALL", choices= ["ALL", "DE", "FR", "ES", "IT"], label="Market")

dbutils.widgets.dropdown("brand", "ALL", ["ALL", "NIVEA", "EUCERIN", "LABELLO", "HANSAPLAST"], "Brand")

In [0]:
report_start_date = dbutils.widgets.get("report_start_date")
report_end_date = dbutils.widgets.get("report_end_date")
market = dbutils.widgets.get("market")
brand = dbutils.widgets.get("brand")

print("Report Start date:", report_start_date)
print("Report end date:", report_end_date)
print("Market:", market)
print("Brand:", brand)

In [0]:
allowed_markets = config["rules"]["allowed_markets"]
allowed_brands = config["rules"]["allowed_brands"]

if market not in allowed_markets:
    raise ValueError(f"Unsupported market: {market}")

if brand not in allowed_brands:
    raise ValueError(f"Unsupported brand: {brand}")

In [0]:
catalog_name = config["catalog"]
schema_name = config["schema"]
landing_volume = config["volumes"]["landing"]
sales_subpath = config["paths"]["sales_subpath"]
inventory_subpath = config["paths"]["inventory_subpath"]

landing_path = f"/Volumes/{catalog_name}/{schema_name}/{landing_volume}"
landing_sales_path = f"/Volumes/{catalog_name}/{schema_name}/{landing_volume}/{sales_subpath}"
landing_inventory_path = f"/Volumes/{catalog_name}/{schema_name}/{landing_volume}/{inventory_subpath}"

bronze_sales_table = f"{catalog_name}.{schema_name}.{config['tables']['bronze_sales']}"

source_path= "/Workspace/Repos/adb-tetiana@startsteps.org/beiersdorf-live-demo/sample_data"

print("Landing path:", landing_path)
print("Bronze table:", bronze_table)
     

In [0]:
dbutils.fs.mkdirs(landing_sales_path)
dbutils.fs.mkdirs(landing_inventory_path)

files = dbutils.fs.ls(source_path)

for file in files:
    if file.name.startswith("sales"):
        dbutils.fs.cp(file.path, f"{landing_sales_path}/{file.name}")
    else:
        dbutils.fs.cp(file.path, f"{landing_inventory_path}/{file.name}")

In [0]:
print("Sales landing files:")
display(dbutils.fs.ls(landing_sales_path))

print("Inventory landing files:")
display(dbutils.fs.ls(landing_inventory_path))

In [0]:
raw_sales_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(landing_sales_path)
)

display(raw_sales_df)

##Filter sales data using the widget inputs

In [0]:
filtered_sales_df = raw_sales_df.filter(
    (F.col("report_date") >= report_start_date) &
    (F.col("report_date") <= report_end_date)
)

if market != "ALL":
    filtered_sales_df = filtered_sales_df.filter(F.col("market") == market)

if brand != "ALL":
    filtered_sales_df = filtered_sales_df.filter(F.col("brand") == brand)

display(filtered_sales_df)


## Print row count and run context

In [0]:
filtered_sales_count = filtered_sales_df.count()

print(f"Report date range used: {report_start_date} to {report_end_date}")
print("Market used:", market)
print("Brand used:", brand)
print("Rows after filtering:", filtered_sales_count)
print("Bronze sales table target:", bronze_sales_table)

## Write the bronze sales table

In [0]:
(
    filtered_sales_df
    .withColumn("ingest_ts", F.current_timestamp())
    .write
    .mode("append") # append is the correct option for bronze tables
    .format("delta")
    .saveAsTable(bronze_sales_table)
)
#/**************************************
#* 'append':
#    * good for bronze history
#    * more realistic for raw ingestion
#    * but reruns may duplicate data
#**/
#* ************************************/

In [0]:
display(spark.read.table(bronze_sales_table))